# `RunnableBindingBase`

`RunnableBindingBase` is a serializable wrapper around another `Runnable`.

It binds keyword arguments, runtime configuration, configuration factories, and optional custom input or output types while delegating execution to the wrapped `Runnable`.

## Inheritance

```python
class RunnableBindingBase(RunnableSerializable[Input, Output]): # Serializable wrapper around another Runnable
```

## Fields

```python
bound: Runnable[Input, Output] # Underlying Runnable receiving delegated operations
kwargs: Mapping[str, Any] # Keyword arguments automatically supplied during execution
config: RunnableConfig # Configuration merged into runtime configuration
config_factories: list[Callable[[RunnableConfig], RunnableConfig]] # Functions that generate additional configuration
custom_input_type: Any | None # Optional replacement input type
custom_output_type: Any | None # Optional replacement output type
```

## Constructor

```python
RunnableBindingBase(
    *,
    bound: Runnable[Input, Output], # Underlying Runnable receiving delegated calls
    kwargs: Mapping[str, Any] | None = None, # Keyword arguments bound to the Runnable
    config: RunnableConfig | None = None, # Configuration bound to the Runnable
    config_factories: list[Callable[[RunnableConfig], RunnableConfig]] | None = None, # Functions that derive additional configuration
    custom_input_type: type[Input] | BaseModel | None = None, # Optional replacement input type
    custom_output_type: type[Output] | BaseModel | None = None, # Optional replacement output type
    **other_kwargs: Any, # Additional model fields
) -> None # Initialize the binding wrapper
```

## Configuration Behaviour

The stored configuration is merged with call-time configuration.

Each configuration factory receives the merged configuration and may return additional configuration values.

Call-time keyword arguments override keyword arguments stored in `kwargs`.

## Overridden Properties and Methods

### `get_name`

Returns the name of the wrapped `Runnable`.

### `InputType`

Returns `custom_input_type` when provided; otherwise, returns the wrapped `Runnable` input type.

### `OutputType`

Returns `custom_output_type` when provided; otherwise, returns the wrapped `Runnable` output type.

### `get_input_schema`

Returns a schema based on `custom_input_type` or delegates schema generation to the wrapped `Runnable`.

### `get_output_schema`

Returns a schema based on `custom_output_type` or delegates schema generation to the wrapped `Runnable`.

### `config_specs`

Returns the configurable-field specifications exposed by the wrapped `Runnable`.

### `get_graph`

Returns the execution graph of the wrapped `Runnable` using merged configuration.

### `is_lc_serializable`

Marks the class as serializable by LangChain.

### `get_lc_namespace`

Returns the LangChain serialization namespace for Runnables.

### `invoke`

Synchronously invokes the wrapped `Runnable` with merged configuration and keyword arguments.

### `ainvoke`

Asynchronously invokes the wrapped `Runnable` with merged configuration and keyword arguments.

### `batch`

Runs synchronous batch execution through the wrapped `Runnable`.

### `abatch`

Runs asynchronous batch execution through the wrapped `Runnable`.

### `batch_as_completed`

Yields indexed synchronous batch results as they complete.

### `abatch_as_completed`

Asynchronously yields indexed batch results as they complete.

### `stream`

Synchronously streams output from the wrapped `Runnable`.

### `astream`

Asynchronously streams output from the wrapped `Runnable`.

### `stream_events`

Synchronously streams execution events from the wrapped supported `Runnable`.

### `astream_events`

Asynchronously streams execution events from the wrapped `Runnable`.

### `transform`

Transforms a synchronous input iterator through the wrapped `Runnable`.

### `atransform`

Transforms an asynchronous input iterator through the wrapped `Runnable`.

In [ ]:
from langchain_core.runnables import RunnableLambda # Import RunnableLambda
from langchain_core.runnables.base import RunnableBindingBase # Import binding base class


def add_prefix(text: str, prefix: str) -> str: # Accept input text and a bound prefix
    return prefix + text # Return prefixed text


class PrefixBinding(RunnableBindingBase[str, str]): # Create a custom binding class
    def __init__(self, bound, prefix: str): # Accept the Runnable and custom prefix
        super().__init__( # Initialize RunnableBindingBase
            bound=bound, # Store the underlying Runnable
            kwargs={"prefix": prefix}, # Bind prefix to every execution
        )


base_runnable = RunnableLambda(add_prefix) # Wrap the function as a Runnable

runnable = PrefixBinding( # Create the custom binding
    bound=base_runnable, # Provide the underlying Runnable
    prefix="Result: ", # Permanently bind the prefix
)

output = runnable.invoke("Hello LangChain") # Execute without supplying prefix again

print(output) # Display the result